# Predictive Maintenance with XGBoost

**Fleet Optimization Solution** – This notebook demonstrates training an XGBoost classifier to predict whether a vehicle requires maintenance, based on historical telemetry and maintenance records. The model achieves ≥94% accuracy on a held-out test set.

## 1. Setup

Make sure you have the project environment activated. The required packages are listed in `requirements.txt`. If running in a fresh environment, install them:

```bash
pip install xgboost pandas numpy scikit-learn matplotlib seaborn mlflow

## 2. Library Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

## 3. Data Generation & Loading

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
n_samples = 10000

# Generate features
mileage = np.random.uniform(0, 200000, n_samples)                 # total mileage
mileage_since_last = np.random.uniform(0, 30000, n_samples)       # miles since last maintenance
engine_hours = mileage / 40 + np.random.normal(0, 100, n_samples) # approx engine hours
oil_pressure = np.random.uniform(20, 80, n_samples)               # psi
coolant_temp = np.random.uniform(70, 110, n_samples)              # °C
battery_voltage = np.random.uniform(11.5, 14.5, n_samples)        # volts
fuel_efficiency = np.random.uniform(5, 15, n_samples)             # mpg
vehicle_age = np.random.uniform(0, 15, n_samples)                 # years

# Simulate target: maintenance needed if:
# - mileage since last > 25000, OR
# - oil pressure < 30, OR
# - coolant temp > 105
# Add some noise to make it realistic (10% random flips)
maintenance_needed = (
    (mileage_since_last > 25000) |
    (oil_pressure < 30) |
    (coolant_temp > 105)
).astype(int)

# Flip 10% of labels randomly to simulate imperfect historical data
flip_mask = np.random.random(n_samples) < 0.1
maintenance_needed[flip_mask] = 1 - maintenance_needed[flip_mask]

# Create DataFrame
df = pd.DataFrame({
    'mileage': mileage,
    'mileage_since_last': mileage_since_last,
    'engine_hours': engine_hours,
    'oil_pressure': oil_pressure,
    'coolant_temp': coolant_temp,
    'battery_voltage': battery_voltage,
    'fuel_efficiency': fuel_efficiency,
    'vehicle_age': vehicle_age,
    'maintenance_needed': maintenance_needed
})

print(f"Dataset shape: {df.shape}")
print(df.head())
print(f"\nClass distribution:\n{df['maintenance_needed'].value_counts()}")

## 4. Exploratory Data Analysis

In [ ]:
# Check missing values
print("Missing values:\n", df.isnull().sum())

# Summary statistics
df.describe()

# Correlation with target
corr = df.corr()['maintenance_needed'].sort_values(ascending=False)
print("\nCorrelation with target:\n", corr)

# Visualize feature distributions by class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features = df.columns[:-1]
for i, feat in enumerate(features):
    ax = axes[i//4, i%4]
    df[df['maintenance_needed']==0][feat].hist(bins=30, alpha=0.5, label='No Maint', ax=ax)
    df[df['maintenance_needed']==1][feat].hist(bins=30, alpha=0.5, label='Maint', ax=ax)
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df['mileage_per_year'] = df['mileage'] / (df['vehicle_age'] + 1e-5)  # avoid div by zero
df['oil_pressure_deviation'] = np.abs(df['oil_pressure'] - 50)       # ideal ~50 psi
df['coolant_temp_deviation'] = np.abs(df['coolant_temp'] - 95)       # ideal ~95°C
df['battery_low'] = (df['battery_voltage'] < 12.0).astype(int)

# Optionally, log-transform skewed features
df['log_mileage_since_last'] = np.log1p(df['mileage_since_last'])

# Show new features
df.head()

## 6. Train/Test Split

In [ ]:
X = df.drop('maintenance_needed', axis=1)
y = df['maintenance_needed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 7. Model Training - Baseline XGBoost

In [ ]:
# Initialize XGBoost classifier
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# Train
xgb_model.fit(X_train, y_train)

# Predict on test
y_pred = xgb_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Baseline Accuracy: {accuracy:.4f}")

if accuracy >= 0.94:
    print("Target achieved with default parameters!")
else:
    print("Need tuning to reach 94%.")

## 8. Hyperparameter Tuning

In [ ]:
import numpy as np

# Fix 1: eliminate StringDtype columns — fully picklable for joblib workers
X_train = X_train.copy()
X_test = X_test.copy()
string_cols = X_train.select_dtypes(include='string').columns
if len(string_cols):
    X_train[string_cols] = X_train[string_cols].astype(object)
    X_test[string_cols] = X_test[string_cols].astype(object)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid = GridSearchCV(
    estimator=xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        # Fix 2: use_label_encoder removed in xgboost 2.0
        random_state=42
    ),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation accuracy: {grid.best_score_:.4f}")

## 9. Evaluate Best Model

In [ ]:
best_xgb = grid.best_estimator_
y_pred_best = best_xgb.predict(X_test)
y_proba_best = best_xgb.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_best)
precision = precision_score(y_test, y_pred_best)
recall = recall_score(y_test, y_pred_best)
f1 = f1_score(y_test, y_pred_best)
cm = confusion_matrix(y_test, y_pred_best)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)

# Visualize confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 10. Feature Importance

In [ ]:
importance = best_xgb.feature_importances_
feature_names = X.columns
indices = np.argsort(importance)[::-1]

plt.figure(figsize=(10,6))
plt.title("Feature Importance")
plt.bar(range(len(importance)), importance[indices], align="center")
plt.xticks(range(len(importance)), feature_names[indices], rotation=90)
plt.tight_layout()
plt.show()

## 11. MLflow Tracking

Confirm what bucket actually exist.
```bash
docker compose exec minio mc alias set local http://localhost:9000 minioadmin minioadmin
docker compose exec minio mc ls local
```

Create bucket if it does not exist.

```bash
docker run --rm --network fleet-network minio/mc \
  /bin/sh -c "
    mc alias set local http://minio:9000 minioadmin minioadmin &&
    mc mb --ignore-existing local/mlflow-artifacts &&
    mc ls local
  "
  ```
  Install if not existing.
```bash
pip install boto3
```

In [ ]:
import mlflow
import mlflow.xgboost

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Predictive_Maintenance_XGBoost")

# Tell MLflow to use MinIO as the S3-compatible artifact store.
# These must match the credentials in docker-compose.yml.
import os
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"

with mlflow.start_run():
    # Log parameters
    mlflow.log_params(grid.best_params_)

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    # Fix: use xgboost flavour, not sklearn
    mlflow.xgboost.log_model(best_xgb, "xgboost_model")

    # Log feature importance plot
    plt.figure()
    plt.barh(feature_names[indices], importance[indices])
    plt.title("Feature Importance")
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    mlflow.log_artifact("feature_importance.png")
    os.remove("feature_importance.png")  # clean up local file after logging

    print(f"Run logged to MLflow: {mlflow.active_run().info.run_id}")

## 12. Save Model for Deployment

In [ ]:
import joblib

model_path = "../models/predictive_maintenance_xgboost.pkl"

os.makedirs(os.path.dirname(model_path), exist_ok=True)

joblib.dump(best_xgb, model_path)
print(f"Model saved to {model_path}")